In [ ]:
!nvidia-smi

Mon Jul 20 07:43:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q -U \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    pypdf \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import torch

from google.colab import files
from pypdf import PdfReader
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer

In [ ]:
uploaded = files.upload()

Saving text_medical.pdf to text_medical (1).pdf


In [ ]:
pdf_path = "/content/text_medical.pdf"

In [ ]:
def extract_pdf_text(pdf_file: str) -> str:
    """Extract text from every readable page of a PDF."""

    reader = PdfReader(pdf_file)
    extracted_pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text()

            if page_text and page_text.strip():
                extracted_pages.append(page_text.strip())
            else:
                print(f"Warning: No text extracted from page {page_number}")

        except Exception as error:
            print(f"Error reading page {page_number}: {error}")

    return "\n\n".join(extracted_pages)


raw_text = extract_pdf_text(pdf_path)

print("Total characters:", len(raw_text))
print(raw_text[:2000])

Total characters: 191750
A village health worker is a person who helps guide family members and neighbors toward better 
health. This person is often chosen by the community because he or she is kind, capable, and 
trustworthy. Some village health workers receive formal training and support from government 
programs such as the Ministry of Health. Others do not have any official title but are respected 
members of the community who help people with health problems. Many of them learn by 
observing others, assisting experienced workers, and studying on their own. 
In a broader sense, a village health worker is anyone who contributes to making the village a 
healthier place. Parents can teach their children the importance of cleanliness. Farmers can 
cooperate to grow enough nutritious food. Teachers can educate students about preventing and 
treating common illnesses and injuries. Children can share health knowledge with their families. 
Shopkeepers can learn about the correct use of me

In [ ]:
def clean_medical_text(text: str) -> str:
    """Clean PDF extraction artifacts without removing medical information."""

    # Remove null characters
    text = text.replace("\x00", " ")

    # Fix words broken across lines with a hyphen
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Convert line breaks inside paragraphs to spaces
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Normalize repeated blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Normalize spaces and tabs
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


cleaned_text = clean_medical_text(raw_text)

print("Characters after cleaning:", len(cleaned_text))
print(cleaned_text[:2000])

Characters after cleaning: 189547
A village health worker is a person who helps guide family members and neighbors toward better health. This person is often chosen by the community because he or she is kind, capable, and trustworthy. Some village health workers receive formal training and support from government programs such as the Ministry of Health. Others do not have any official title but are respected members of the community who help people with health problems. Many of them learn by observing others, assisting experienced workers, and studying on their own. In a broader sense, a village health worker is anyone who contributes to making the village a healthier place. Parents can teach their children the importance of cleanliness. Farmers can cooperate to grow enough nutritious food. Teachers can educate students about preventing and treating common illnesses and injuries. Children can share health knowledge with their families. Shopkeepers can learn about the correct use of med

In [ ]:
with open("cleaned_medical_text.txt", "w", encoding="utf-8") as file:
    file.write(cleaned_text)

print("Cleaned text saved.")

Cleaned text saved.


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.
Vocabulary size: 151665


In [ ]:
def create_token_chunks(
    text: str,
    tokenizer,
    chunk_size: int = 768,
    overlap: int = 80,
) -> list[str]:
    """
    Divide text into overlapping token chunks.

    chunk_size:
        Maximum number of tokens in one sample.

    overlap:
        Number of tokens repeated between consecutive chunks.
    """

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    chunks = []
    step_size = chunk_size - overlap

    for start in range(0, len(token_ids), step_size):
        end = start + chunk_size
        current_ids = token_ids[start:end]

        # Skip extremely small final chunks
        if len(current_ids) < 100:
            continue

        chunk_text = tokenizer.decode(
            current_ids,
            skip_special_tokens=True,
        )

        chunks.append(chunk_text.strip())

    return chunks


text_chunks = create_token_chunks(
    cleaned_text,
    tokenizer,
    chunk_size=768,
    overlap=80,
)

print("Number of chunks:", len(text_chunks))
print("\nFirst chunk:\n")
print(text_chunks[0][:2000])

Number of chunks: 55

First chunk:

A village health worker is a person who helps guide family members and neighbors toward better health. This person is often chosen by the community because he or she is kind, capable, and trustworthy. Some village health workers receive formal training and support from government programs such as the Ministry of Health. Others do not have any official title but are respected members of the community who help people with health problems. Many of them learn by observing others, assisting experienced workers, and studying on their own. In a broader sense, a village health worker is anyone who contributes to making the village a healthier place. Parents can teach their children the importance of cleanliness. Farmers can cooperate to grow enough nutritious food. Teachers can educate students about preventing and treating common illnesses and injuries. Children can share health knowledge with their families. Shopkeepers can learn about the correct use of m

In [ ]:
def format_training_chunk(chunk: str) -> str:
    return (
        "Medical educational reference material:\n\n"
        f"{chunk}"
        f"{tokenizer.eos_token}"
    )


formatted_chunks = [
    format_training_chunk(chunk)
    for chunk in text_chunks
]

In [ ]:
dataset = Dataset.from_dict({
    "text": formatted_chunks
})

print(dataset)
print(dataset[0]["text"][:1500])

Dataset({
    features: ['text'],
    num_rows: 55
})
Medical educational reference material:

A village health worker is a person who helps guide family members and neighbors toward better health. This person is often chosen by the community because he or she is kind, capable, and trustworthy. Some village health workers receive formal training and support from government programs such as the Ministry of Health. Others do not have any official title but are respected members of the community who help people with health problems. Many of them learn by observing others, assisting experienced workers, and studying on their own. In a broader sense, a village health worker is anyone who contributes to making the village a healthier place. Parents can teach their children the importance of cleanliness. Farmers can cooperate to grow enough nutritious food. Teachers can educate students about preventing and treating common illnesses and injuries. Children can share health knowledge with their

In [ ]:
if len(dataset) >= 10:
    dataset_split = dataset.train_test_split(
        test_size=0.10,
        seed=42,
    )

    train_dataset = dataset_split["train"]
    eval_dataset = dataset_split["test"]

else:
    train_dataset = dataset
    eval_dataset = None

print("Training samples:", len(train_dataset))

if eval_dataset is not None:
    print("Validation samples:", len(eval_dataset))

Training samples: 49
Validation samples: 6


In [ ]:
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Compute dtype:", compute_dtype)

Compute dtype: torch.bfloat16


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

print("Base model loaded successfully.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully.


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
output_directory = "/content/medical-qwen-lora"

training_config = SFTConfig(
    output_dir=output_directory,

    # Training
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    # Sequence configuration
    max_length=768,
    packing=False,

    # Optimisation
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    max_grad_norm=0.3,

    # Precision
    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),

    # Gradient checkpointing reduces GPU memory use
    gradient_checkpointing=True,

    # Logging and saving
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,

    # Evaluation
    eval_strategy="epoch" if eval_dataset is not None else "no",

    # Dataset
    dataset_text_field="text",

    # Other
    report_to="none",
    seed=42,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Trainer created.")

Adding EOS to train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/49 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/6 [00:00<?, ? examples/s]

Trainer created.


In [ ]:
trainer.model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [ ]:
train_result = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,2.204737,1.998815,2.013826,37376.000000,0.527814
2,2.002152,1.937726,1.864648,74752.000000,0.532812
3,1.847159,1.927356,1.876706,112128.000000,0.536071


In [ ]:
print(train_result)

TrainOutput(global_step=21, training_loss=1.9887447357177734, metrics={'train_runtime': 687.0604, 'train_samples_per_second': 0.214, 'train_steps_per_second': 0.031, 'total_flos': 893977735200768.0, 'train_loss': 1.9887447357177734, 'epoch': 3.0})


In [ ]:
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

***** train metrics *****
  epoch                    =        3.0
  total_flos               =   832581GF
  train_loss               =     1.9887
  train_runtime            = 0:11:27.06
  train_samples_per_second =      0.214
  train_steps_per_second   =      0.031


In [ ]:
if eval_dataset is not None:
    evaluation_results = trainer.evaluate()

    trainer.log_metrics("eval", evaluation_results)
    trainer.save_metrics("eval", evaluation_results)

    print(evaluation_results)

Training Loss,Validation Loss,Epoch,Entropy,Num Tokens,Mean Token Accuracy
1.847159,1.927356,3,1.876706,112128.000000,0.536071


***** eval metrics *****
  eval_entropy             =   1.8767
  eval_loss                =   1.9274
  eval_mean_token_accuracy =   0.5361
  eval_num_tokens          = 112128.0
{'eval_loss': 1.9273563623428345, 'eval_entropy': 1.876705805460612, 'eval_num_tokens': 112128.0, 'eval_mean_token_accuracy': 0.5360712756713232}


In [ ]:
final_adapter_path = "/content/final-medical-lora-adapter"

trainer.model.save_pretrained(final_adapter_path)
tokenizer.save_pretrained(final_adapter_path)

print("LoRA adapter saved at:", final_adapter_path)

LoRA adapter saved at: /content/final-medical-lora-adapter


In [ ]:
!ls -lh /content/final-medical-lora-adapter

total 47M
-rw-r--r-- 1 root root 1.1K Jul 20 08:11 adapter_config.json
-rw------- 1 root root  36M Jul 20 08:11 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Jul 20 08:11 chat_template.jinja
-rw-r--r-- 1 root root 5.1K Jul 20 08:11 README.md
-rw-r--r-- 1 root root  694 Jul 20 08:11 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jul 20 08:11 tokenizer.json


In [ ]:
!zip -r medical-lora-adapter.zip /content/final-medical-lora-adapter

  adding: content/final-medical-lora-adapter/ (stored 0%)
  adding: content/final-medical-lora-adapter/adapter_model.safetensors (deflated 22%)
  adding: content/final-medical-lora-adapter/chat_template.jinja (deflated 71%)
  adding: content/final-medical-lora-adapter/tokenizer_config.json (deflated 59%)
  adding: content/final-medical-lora-adapter/adapter_config.json (deflated 59%)
  adding: content/final-medical-lora-adapter/tokenizer.json (deflated 81%)
  adding: content/final-medical-lora-adapter/README.md (deflated 65%)


In [ ]:
model = trainer.model
model.eval()
model.config.use_cache = True

In [ ]:
def ask_medical_model(
    question: str,
    max_new_tokens: int = 200,
) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You provide educational information based on the supplied "
                "medical reference material. Do not diagnose patients. "
                "For emergencies or serious symptoms, recommend consultation "
                "with a qualified healthcare professional."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    return answer.strip()

In [ ]:
question = "What is the difference between infectious and non-infectious diseases?"

answer = ask_medical_model(question)

print("Question:", question)
print("\nAnswer:\n", answer)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Question: What is the difference between infectious and non-infectious diseases?

Answer:
 Infectious disease refers to conditions caused by microorganisms such as bacteria, viruses, fungi, or parasites that can be transmitted from one person to another through contact, air, water, food, or other bodily fluids. These infections often have specific incubation periods and may cause mild illnesses like colds, flu, typhoid fever, cholera, malaria, tuberculosis, measles, smallpox, tetanus, diphtheria, whooping cough, yellow fever, bubonic plague, leprosy, syphilis, gonorrhea, hepatitis A, B, C, D, E, HIV/AIDS, rabies, meningitis, pneumonia, typhus, dysentery, diarrhea, scabies, ringworm, warts, athlete’s foot, chicken pox, shingles, boils, sores, ulcers, rashes, blisters, scars, bruises, sprains, fractures, bites, cuts, scrapes, burns, insect stings,


In [ ]:
print(
    ask_medical_model(
        "Why do antibiotics not work against most viral infections?"
    )
)

Antibiotics are effective against bacterial infections because they interfere with bacterial cell growth and reproduction. However, viruses cannot be killed by antibiotics because they lack cellular structures that allow for such interference. Therefore, antibiotics are generally ineffective against viral infections such as colds, flu, chickenpox, measles, mumps, HIV/AIDS, hepatitis, mononucleosis, and many other illnesses caused by viruses. Some antibiotics may help reduce symptoms in certain viral infections but will not cure them. It is important to distinguish between bacterial and viral infections correctly so appropriate treatment can be given. Antibiotics should only be used when prescribed by a doctor. Overuse of antibiotics can lead to antibiotic resistance, where bacteria become resistant to certain drugs, making it more difficult to treat serious infections in the future. In general, antibiotics are less effective against viral infections than against bacterial ones, althoug

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

In [ ]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/final-medical-lora-adapter"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
    else torch.float16
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
)

model.eval()

print("Fine-tuned adapter loaded.")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Fine-tuned adapter loaded.
